# **Análisis textual y procesamiento de corpus**

# 1️⃣ Instalación de librerías necesarias:

In [ ]:
%pip install requests beautifulsoup4 nltk matplotlib pandas wikipedia-api

# 2️⃣ Importar librerías

In [ ]:
import requests
from bs4 import BeautifulSoup
import nltk
from nltk.corpus import stopwords
from nltk import word_tokenize, sent_tokenize
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd
import re
import math
import wikipediaapi

# 3️⃣ Scrapeando texto: la Wikipedia

El **web scraping** es la técnica que permite extraer información de páginas web de forma automatizada.  

---

### Conceptos clave

1. **Requests:** permite enviar solicitudes HTTP a una página web y obtener su contenido HTML.
2. **BeautifulSoup:** se encarga de parsear el HTML y navegar por la estructura del documento para extraer el texto o los elementos que nos interesen.
3. **Precauciones legales y éticas:**
   - Siempre respeta el archivo `robots.txt` del sitio web.
   - No realices scraping masivo sin permiso.
   - Para Wikipedia, es recomendable usar la **API oficial** o declarar un **User-Agent** identificativo de tu script.


In [ ]:
def scrape_text(url):
    headers = {
        # Configuramos el user-agent para evitar bloqueos
        "User-Agent": "CursoIR2025/1.0 (https://esei.uvigo.es; alumno@uvigo.es)"
    }
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")

    # Eliminar scripts, estilos y basura
    for script in soup(["script", "style"]):
        script.extract()
    text = soup.get_text(separator=" ")
    text = re.sub(r"\s+", " ", text)
    return text


# Ejemplo: URLs de RI para la prueba
url_en = "https://en.wikipedia.org/wiki/Information_retrieval"
url_es = "https://es.wikipedia.org/wiki/B%C3%BAsqueda_y_recuperación_de_información"

texto_en = scrape_text(url_en)
texto_es = scrape_text(url_es)

print("English text (preview):", texto_en[:500])
print("Español text (preview):", texto_es[:500])

Para Wikipedia, es recomendable usar la **API oficial** o declarar un User-Agent identificativo de tu script.

In [ ]:
def obtener_texto_wiki(titulo_es, debug=False):

    user_agent = "CursoIR2025/1.0 (https://esei.uvigo.es; alumno@uvigo.es)"

    wiki_es = wikipediaapi.Wikipedia(
        language="es",
        extract_format=wikipediaapi.ExtractFormat.WIKI,
        user_agent=user_agent,
    )

    wiki_en = wikipediaapi.Wikipedia(
        language="en",
        extract_format=wikipediaapi.ExtractFormat.WIKI,
        user_agent=user_agent,
    )
    # Página en español
    page_es = wiki_es.page(titulo_es)
    if not page_es.exists():
        if debug:
            return f"La página '{titulo_es}' no existe en español"
    else:
        if debug:
            print(f"Página encontrada: {titulo_es} (es)")

    # Intentamos encontrar el enlace al artículo en inglés
    langlink_en = page_es.langlinks.get("en")

    if langlink_en:
        titulo_en = langlink_en.title
        page_en = wiki_en.page(titulo_en)
        texto_en = page_en.text if page_en.exists() else ""
    else:
        texto_en = ""  # no existe versión en inglés

    if debug:
        print("=== Español ===")
        print(f"Título: {titulo_es} | Longitud: {len(page_es.text)}")
        print("=== Inglés ===")
        if texto_en:
            print(f"Título: {titulo_en} | Longitud: {len(texto_en)}")
        else:
            print("No hay versión en inglés.")

    return page_es.text, texto_en


# Ejemplo: obtenemos artículos sobre Inteligencia Artificial, Aprendizaje automático y Búsqueda y recuperación de información.
titulos = [
    "Inteligencia artificial",
    "Aprendizaje automático",
    "Búsqueda y recuperación de información",
]
# Listas para acumular los textos
textos_es = []
textos_en = []

for t in titulos:
    texto_es, texto_en = obtener_texto_wiki(t, debug=True)
    textos_es.append(texto_es)
    textos_en.append(texto_en)

# Concatenamos todo el texto
texto_total_es = "\n".join(textos_es)
texto_total_en = "\n".join(textos_en)

print("\nFragmento del texto combinado en es:")
print(texto_total_es[:800])
print("\nFragmento del texto combinado en en:")
print(texto_total_en[:800])

# 4️⃣ Tokenización

La tokenización en unidades léxicas, consiste en dividir el texto en unidades más pequeñas (que a menudo coinciden con palabras). A pesar de que se trata de una tarea muy básica y necesaria para poder llevar a cabo tareas de análisis más avanzadas, esta tarea presenta numerosos problemas que no son fáciles de solucionar. En este apartado aprenderemos algunas técnicas para tokenizar el texto en unidades léxicas e iremos observando los diferentes problemas que aparecen y cómo se pueden solucionar. Las pruebas de los diferentes sistemas las haremos con la primera oración del documento sobre Inteligencia Artificial:

*La inteligencia artificial (abreviado: IA), en el contexto de las ciencias de la computación, es una disciplina y un conjunto de capacidades cognoscitivas e intelectuales expresadas por sistemas informáticos o combinaciones de algoritmos cuyo propósito es la creación de máquinas que imiten la inteligencia humana.*

A pesar de que el resultado deseado de la tokenización puede depender de las tareas concretas, la salida deseada de nuestro sistema tendría que ser algo del siguiente estilo:

*['La', 'inteligencia', 'artificial', '(', 'abreviado', ':', 'IA', ')', ',', 'en', 'el', 'contexto', 'de', 'las', 'ciencias', 'de', 'la', 'computación',',', 'es', 'una', 'disciplina', 'y', 'un', 'conjunto', 'de', 'capacidades', 'cognoscitivas', 'e', 'intelectuales', 'expresadas', 'por', 'sistemas', 'informáticos', 'o', 'combinaciones', 'de', 'algoritmos', 'cuyo', 'propósito', 'es', 'la', 'creación', 'de', 'máquinas', 'que', 'imiten', 'la', 'inteligencia', 'humana', '.']*

El **primer tokenizador** que probaremos utiliza la instrucción split(), que divide una cadena según el separador que se indique, y si no se indica nada, por espacios. Lo podemos ver a continuación:

In [ ]:
import nltk

oracion = "La inteligencia artificial (abreviado: IA), en el contexto de las ciencias de la computación, es una disciplina y un conjunto de capacidades cognoscitivas e intelectuales expresadas por sistemas informáticos o combinaciones de algoritmos cuyo propósito es la creación de máquinas que imiten la inteligencia humana."

tokens = oracion.split()
print(tokens)

que nos da la siguiente salida, que no es exactamente la que queríamos:

```
['La', 'inteligencia', 'artificial', '(abreviado:', 'IA),', 'en', 'el', 'contexto', 'de', 'las', 'ciencias', 'de', 'la', 'computación,', 'es', 'una', 'disciplina', 'y', 'un', 'conjunto', 'de', 'capacidades', 'cognoscitivas', 'e', 'intelectuales', 'expresadas', 'por', 'sistemas', 'informáticos', 'o', 'combinaciones', 'de', 'algoritmos', 'cuyo', 'propósito', 'es', 'la', 'creación', 'de', 'máquinas', 'que', 'imiten', 'la', 'inteligencia', 'humana.']
```

NLTK proporciona una serie de tokenizadores que presentamos a continuación:

### WhitespaceTokenizer

Separa el texto por espacios en blanco, como lo hemos hecho en el ejemplo anterior. En este caso los espacios en blanco pueden ser los caracteres: espacio en blanco, tabulador y nueva línea.

In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")

oracion = "La inteligencia artificial (abreviado: IA), en el contexto de las ciencias de la computación, es una disciplina y un conjunto de capacidades cognoscitivas e intelectuales expresadas por sistemas informáticos o combinaciones de algoritmos cuyo propósito es la creación de máquinas que imiten la inteligencia humana."
tokens = nltk.tokenize.WhitespaceTokenizer().tokenize(oracion)
print(tokens)

Su salida es como la anterior:

```
['La', 'inteligencia', 'artificial', '(abreviado:', 'IA),', 'en', 'el', 'contexto', 'de', 'las', 'ciencias', 'de', 'la', 'computación,', 'es', 'una', 'disciplina', 'y', 'un', 'conjunto', 'de', 'capacidades', 'cognoscitivas', 'e', 'intelectuales', 'expresadas', 'por', 'sistemas', 'informáticos', 'o', 'combinaciones', 'de', 'algoritmos', 'cuyo', 'propósito', 'es', 'la', 'creación', 'de', 'máquinas', 'que', 'imiten', 'la', 'inteligencia', 'humana.']
```

Aprovecho el ejemplo para explicar varias maneras de importar un tokenizador, o cualquier método de una clase determinada. En el caso anterior hemos importado todo el nltk y hemos llamado al método haciendo:

```
tokens=nltk.tokenize.WhitespaceTokenizer().tokenize(oracion)
```

Esto también se puede hacer de la siguiente manera alternativa (fíjate como accedemos ahora al método tokenize):

In [ ]:
from nltk.tokenize import WhitespaceTokenizer

oracion = "La inteligencia artificial (abreviado: IA), en el contexto de las ciencias de la computación, es una disciplina y un conjunto de capacidades cognoscitivas e intelectuales expresadas por sistemas informáticos o combinaciones de algoritmos cuyo propósito es la creación de máquinas que imiten la inteligencia humana."
tokens = WhitespaceTokenizer().tokenize(oracion)
print(tokens)

### SpaceTokenizer

Es igual que el anterior, pero en este caso el único carácter que se tiene en cuenta es el espacio en blanco (" "). Equivale a split(" "). En nuestro ejemplo la salida seria exactamente la misma por lo que no es necesario proporcionar ni el código ni el programa.


### TreebankWordTokenizer

Este tokenizador utiliza las convenciones del Penn Treebank corpus, qué es un corpus anotado del inglés creado en 1980 a partir de artículos del Wall Street Journal. Vamos a probarla con la correspondiente primera oración del texto relativo a Artificial Intelligente:

In [ ]:
from nltk.tokenize import TreebankWordTokenizer as tokenizer

oracion = "Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making."
tokens = tokenizer().tokenize(oracion)
print(tokens)

y la salida será:

```
['Artificial', 'intelligence', '(', 'AI', ')', 'is', 'the', 'capability', 'of', 'computational', 'systems', 'to', 'perform', 'tasks', 'typically', 'associated', 'with', 'human', 'intelligence', ',', 'such', 'as', 'learning', ',', 'reasoning', ',', 'problem-solving', ',', 'perception', ',', 'and', 'decision-making', '.']
```

Este tokenizador se usa habitualmente para el inglés (fue entrenado en este idioma), y por este motivo es el que por defecto se utiliza con la función específica que hace de wrapper (*wrapper function*) para simplificar su uso:


In [ ]:
from nltk.tokenize import word_tokenize

oracion = "Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making."
tokens = word_tokenize(oracion)
print(tokens)

que proporciona exactamente la misma salida.

### RegexpTokenizer

Este es un tipo de tokenizador que nos permite un control total sobre el proceso de tokenización. Claro que para sacar el máximo provecho hay que dominar las expresiones regulares de Python. Podéis encontrar una explicación detallada en w3schools.com (https://www.w3schools.com/python/python_regex.asp).

Veamos algunos ejemplos:

In [ ]:
import re
from nltk.tokenize import RegexpTokenizer

oracion = "La inteligencia artificial (abreviado: IA), en el contexto de las ciencias de la computación, es una disciplina y un conjunto de capacidades cognoscitivas e intelectuales expresadas por sistemas informáticos o combinaciones de algoritmos cuyo propósito es la creación de máquinas que imiten la inteligencia humana."
tokenizador = RegexpTokenizer(r"\s+", gaps=True)
tokens = tokenizador.tokenize(oracion)
print(tokens)

Que da la siguiente salida (que coincide con la del tokenizador por espacios en blanco):

```
['La', 'inteligencia', 'artificial', '(abreviado:', 'IA),', 'en', 'el', 'contexto', 'de', 'las', 'ciencias', 'de', 'la', 'computación,', 'es', 'una', 'disciplina', 'y', 'un', 'conjunto', 'de', 'capacidades', 'cognoscitivas', 'e', 'intelectuales', 'expresadas', 'por', 'sistemas', 'informáticos', 'o', 'combinaciones', 'de', 'algoritmos', 'cuyo', 'propósito', 'es', 'la', 'creación', 'de', 'máquinas', 'que', 'imiten', 'la', 'inteligencia', 'humana.']
```

Si queremos cortar por cualquier cosa que no sea una secuencia de caracteres o de números usaremos lo siguiente:

In [ ]:
from nltk.tokenize import RegexpTokenizer

oracion = "La inteligencia artificial (abreviado: IA), en el contexto de las ciencias de la computación, es una disciplina y un conjunto de capacidades cognoscitivas e intelectuales expresadas por sistemas informáticos o combinaciones de algoritmos cuyo propósito es la creación de máquinas que imiten la inteligencia humana."
tokenizador = RegexpTokenizer(r"\w+|[^\w\s]+")
tokens = tokenizador.tokenize(oracion)
print(tokens)

Que nos ofrece la siguiente salida:

```
['La', 'inteligencia', 'artificial', '(', 'abreviado', ':', 'IA', '),', 'en', 'el', 'contexto', 'de', 'las', 'ciencias', 'de', 'la', 'computación', ',', 'es', 'una', 'disciplina', 'y', 'un', 'conjunto', 'de', 'capacidades', 'cognoscitivas', 'e', 'intelectuales', 'expresadas', 'por', 'sistemas', 'informáticos', 'o', 'combinaciones', 'de', 'algoritmos', 'cuyo', 'propósito', 'es', 'la', 'creación', 'de', 'máquinas', 'que', 'imiten', 'la', 'inteligencia', 'humana', '.']
```

Como podemos observar, nos separa 'abreviado' del '(' y del ':'. Es lo que queríamos obtener. Pero, ¿qué pasaría si la frase fuese?

*Un bot conversacional creado por OpenAI usando sus modelos de lenguaje grande fundacionales GPT-3 y GPT-4.*


In [ ]:
from nltk.tokenize import RegexpTokenizer

oracion = "Un bot conversacional creado por OpenAI usando sus modelos de lenguaje grande fundacionales GPT-3 y GPT-4."
tokenizador = RegexpTokenizer(r"\w+|[^\w\s]+")
tokens = tokenizador.tokenize(oracion)
print(tokens)


 Lo que realmente queremos es tener 'GPT-3' y 'GPT-4' como token. Podríamos solucionar esto modificando la expresión regular de forma que añadimos estas dos unidades como tokens.

 ```
 tokenizador=RegexpTokenizer(r'GPT\-3|GPT\-4|\w+|[^\w\s]+')
 ```

In [ ]:
from nltk.tokenize import RegexpTokenizer

oracion = "Un bot conversacional creado por OpenAI usando sus modelos de lenguaje grande fundacionales GPT-3 y GPT-4."
tokenizador = RegexpTokenizer(r"GPT\-3|GPT\-4|\w+|[^\w\s]+")
tokens = tokenizador.tokenize(oracion)
print(tokens)

que nos devolverá la tokenización correcta de estas dos unidades:

```
['Un', 'bot', 'conversacional', 'creado', 'por', 'OpenAI', 'usando', 'sus', 'modelos', 'de', 'lenguaje', 'grande', 'fundacionales', 'GPT-3', 'y', 'GPT-4', '.']
```

El hecho de añadir una lista de abreviaturas frecuentes es bastante habitual, pero no seamos ilusos. No podemos esperar tener una lista suficiente completa de acrónimos. De ahí que tengamos que intentar expresar los acrónimos de una manera más general, cómo por ejemplo:

```
tokenizador = RegexpTokenizer(r'\w+-\d+|\w+|[^\w\s]+')
```

In [ ]:
from nltk.tokenize import RegexpTokenizer

oracion = "Un bot conversacional creado por OpenAI usando sus modelos de lenguaje grande fundacionales GPT-3 y GPT-4."
tokenizador = RegexpTokenizer(r"\w+-\d+|\w+|[^\w\s]+")
tokens = tokenizador.tokenize(oracion)
print(tokens)

que nos ofrece la misma salida pero que ahora contempla cualquier acrónimo tipo GPT-3.



---

#**Ejercicio:**

Suponer que tenemos una frase con acrónimos del estilo siguiente y queremos usar el tokenizador *RegexpTokenizer*, ¿qué tendría que hacer?

```
oracion="El Sr. J.P. y la Dra. M.L. trabajaron en colaboración con la O.N.U. y la U.E. durante el proyecto de I.A."
```


---



# 5️⃣ Segmentación


En la sección anterior hemos visto como tokenizar, es decir, separar el texto en unidad léxicas. Aquí veremos cómo podemos separar en unidades parecidas a oraciones. Este proceso recibe el nombre de segmentación.

En esta sección usaremos dos segmentos de prueba, uno que no presentará grandes problemas:

```
Los modelos geométricos, construidos en el espacio de instancias y que pueden tener una, dos o múltiples dimensiones. Si hay un borde de decisión lineal entre las clases, se dice que los datos son linealmente separables.
```

y otro que sí que presenta numerosos problemas.

```
El señor J.P. revisó el módulo de la A.P.I. en el servidor a las 09:30h antes de la reunión con el equipo de R.I. Luego la Dra. M.L. ejecutó las pruebas del nuevo O.S. sobre la G.P.U. principal a las 17:45h, justo antes del cierre de las instalaciones.
```

### PunktSentenceTokenizer

Este es un segmentador sencillo que básicamente segmenta por puntos. Para ver cómo funciona probamos lo siguiente:

In [ ]:
from nltk.tokenize import PunktSentenceTokenizer

parrafo1 = "Los modelos geométricos, construidos en el espacio de instancias y que pueden tener una, dos o múltiples dimensiones. Si hay un borde de decisión lineal entre las clases, se dice que los datos son linealmente separables."

parrafo2 = "El señor J.P. revisó el módulo de la A.P.I. en el servidor a las 09:30h antes de la reunión con el equipo de R.I. Luego la Dra. M.L. ejecutó las pruebas del nuevo O.S. sobre la G.P.U. principal a las 17:45h, justo antes del cierre de las instalaciones."

segmentador = PunktSentenceTokenizer()
segmentos1 = segmentador.tokenize(parrafo1)
print(segmentos1)
segmentos2 = segmentador.tokenize(parrafo2)
print(segmentos2)

Que proporciona la siguiente salida:

```
['Los modelos geométricos, construidos en el espacio de instancias y que pueden tener una, dos o múltiples dimensiones.', 'Si hay un borde de decisión lineal entre las clases, se dice que los datos son linealmente separables.']
['El Ing.', 'J.P.', 'revisó el módulo de la A.P.I.', 'en el servidor a las 09:30h antes de la reunión con el equipo de R.I.', 'Luego la Dra.', 'M.L.', 'ejecutó las pruebas del nuevo O.S.', 'sobre la G.P.U.', 'principal a las 17:45h, justo antes del cierre de las instalaciones.']
```

### sent_tokenize()

El proceso de segmentación también se puede hacer con sent_tokenizer(), que llama a una instancia especial del PunktSentenceTokenizer que ha sido entrenado y que funciona bastante bien para varias lenguas europeas. Lo vemos a continuación:

In [ ]:
from nltk.tokenize import sent_tokenize

parrafo1 = "Los modelos geométricos, construidos en el espacio de instancias y que pueden tener una, dos o múltiples dimensiones. Si hay un borde de decisión lineal entre las clases, se dice que los datos son linealmente separables."

parrafo2 = "El señor J.P. revisó el módulo de la A.P.I. en el servidor a las 09:30h antes de la reunión con el equipo de R.I. Luego la Dra. M.L. ejecutó las pruebas del nuevo O.S. sobre la G.P.U. principal a las 17:45h, justo antes del cierre de las instalaciones."

segmentos1 = sent_tokenize(parrafo1)
print(segmentos1)
segmentos2 = sent_tokenize(parrafo2)
print(segmentos2)

Sigue sin ser del todo de nuestro agrado ya que la salida es la siguiente:

```
['Los modelos geométricos, construidos en el espacio de instancias y que pueden tener una, dos o múltiples dimensiones.', 'Si hay un borde de decisión lineal entre las clases, se dice que los datos son linealmente separables.']

['El señor J.P. revisó el módulo de la A.P.I.', 'en el servidor a las 09:30h antes de la reunión con el equipo de R.I. Luego la Dra.', 'M.L.', 'ejecutó las pruebas del nuevo O.S.', 'sobre la G.P.U.', 'principal a las 17:45h, justo antes del cierre de las instalaciones.']
```

### Segmentador concreto para una lengua determinada

Con los datos del NLTK se distribuyen una serie de segmentadores para lenguas determinadas. Concretamente se distribuyen para: checo, finlandés, noruego, español, danés, francés, polaco, sueco, holandés, alemán, portugués, turco, inglés, griego, estonio, italiano, esloveno. Vamos a cargar el específico para español y luego para el inglés:

In [ ]:
import nltk.data

parrafo = "El señor J.P. revisó el módulo de la A.P.I. en el servidor a las 09:30h antes de la reunión con el equipo de R.I. Luego la Dra. M.L. ejecutó las pruebas del nuevo O.S. sobre la G.P.U. principal a las 17:45h, justo antes del cierre de las instalaciones."

segmentador = nltk.data.load("tokenizers/punkt/PY3/spanish.pickle")
segmentos = segmentador.tokenize(parrafo)
print(segmentos)

In [ ]:
import nltk.data

parrafo = (
    "Today Mr. Smith and Ms. Johanson will learn more about A.I on St. Martin's Day."
)

segmentador = nltk.data.load("tokenizers/punkt/PY3/english.pickle")
segmentos = segmentador.tokenize(parrafo)
print(segmentos)

Que proporciona la salida en español:

```
['El señor J.P.', 'revisó el módulo de la A.P.I.', 'en el servidor a las 09:30h antes de la reunión con el equipo de R.I.', 'Luego la Dra.', 'M.L.', 'ejecutó las pruebas del nuevo O.S.', 'sobre la G.P.U.', 'principal a las 17:45h, justo antes del cierre de las instalaciones.']
```

Vemos que para español la salida sigue sin ser fantástica. Y en inglés:

```
["Today Mr. Smith and Ms. Johanson will learn more about A.I on St. Martin's Day."]
```



### Entrenamiento de un segmentador

NLTK implementa el algoritmo de Kiss and Strunk (2006) para entrenar un segmentador a partir de texto sin ningún tipo de anotación. Vamos a entrenar un segmentador para el castellano (de hecho crearemos un `spanish_propio.pickle`) a partir de un documento de la Wikipedia, aunque sería necesario hacerlo a partir de un corpus más completo. Cada línea de ese documento es una oración:

In [ ]:
# Descargar recursos necesarios
from nltk.tokenize.punkt import PunktTrainer
import pickle
import codecs

nltk.download("punkt")

# --- 1. Prepara el corpus ---
texto = """
Los modelos geométricos, construidos en el espacio de instancias y que pueden tener una, dos o múltiples dimensiones.
Si hay un borde de decisión lineal entre las clases, se dice que los datos son linealmente separables.
El señor J.P. revisó el módulo de la A.P.I. a las 09:30h.
Luego la Dra. M.L. ejecutó las pruebas del nuevo O.S. sobre la G.P.U. principal a las 17:45h.
"""
corpus_es = codecs.open("/content/texto_entrenamiento.txt", "r", "utf8").read()

# --- 2. Entrena el modelo ---
trainer = PunktTrainer()
trainer.INCLUDE_ALL_COLLOCS = True
trainer.train(texto)

# --- 3. Crea el tokenizador entrenado ---
tokenizador_entrenado = PunktSentenceTokenizer(trainer.get_params())
# Guarda el tokenizador entrenado en un archivo
with open("spanish_propio.pickle", "wb") as f:
    pickle.dump(tokenizador_entrenado, f)

print("Tokenizador guardado en tokenizador_es.pkl")

In [ ]:
import pickle

# Cargar tokenizador desde el archivo .pkl
with open("/content/spanish_propio.pickle", "rb") as f:
    segmentador = pickle.load(f)

# Probamos con tus párrafos
parrafo1 = "Los modelos geométricos, construidos en el espacio de instancias y que pueden tener una, dos o múltiples dimensiones. Si hay un borde de decisión lineal entre las clases, se dice que los datos son linealmente separables."

parrafo2 = "El señor J.P. revisó el módulo de la A.P.I. en el servidor a las 09:30h antes de la reunión con el equipo de R.I. Luego la Dra. M.L. ejecutó las pruebas del nuevo O.S. sobre la G.P.U. principal a las 17:45h, justo antes del cierre de las instalaciones."

segmentos1 = segmentador.tokenize(parrafo1)
print(segmentos1)

segmentos2 = segmentador.tokenize(parrafo2)
print(segmentos2)

Que ofrece la siguiente salida:

```
['Los modelos geométricos, construidos en el espacio de instancias y que pueden tener una, dos o múltiples dimensiones.', 'Si hay un borde de decisión lineal entre las clases, se dice que los datos son linealmente separables.']
['El señor J.P. revisó el módulo de la A.P.I. en el servidor a las 09:30h antes de la reunión con el equipo de R.I.', 'Luego la Dra.', 'M.L. ejecutó las pruebas del nuevo O.S. sobre la G.P.U. principal a las 17:45h, justo antes del cierre de las instalaciones.']
```

**¿Funciona del todo correctamente el segmentador que hemos entrenado?**

### Personaliza el segmentador

Podemos personalizar el segmentador entrenado para añadir nuevas abreviaturas o acrónimos que no han sido detectados en el proceso de entrenamiento. Lo podemos hacer específicamente para un programa determinado. En el `spanish_propio.pickle` que hemos entrenado vamos a añadir una serie de abreviaturas (fíjate que lo hacemos en minúsculas y sin el punto final):

In [ ]:
import nltk.data

# Cargar tokenizador desde el archivo .pkl
with open("/content/spanish_propio.pickle", "rb") as f:
    segmentador = pickle.load(f)

abreviaturas_extra = ["dra"]
segmentador._params.abbrev_types.update(abreviaturas_extra)

# Probamos con tus párrafos
parrafo1 = "Los modelos geométricos, construidos en el espacio de instancias y que pueden tener una, dos o múltiples dimensiones. Si hay un borde de decisión lineal entre las clases, se dice que los datos son linealmente separables."

parrafo2 = "El señor J.P. revisó el módulo de la A.P.I. en el servidor a las 09:30h antes de la reunión con el equipo de R.I. Luego la Dra. M.L. ejecutó las pruebas del nuevo O.S. sobre la G.P.U. principal a las 17:45h, justo antes del cierre de las instalaciones."

segmentos1 = segmentador.tokenize(parrafo1)
print(segmentos1)
segmentos2 = segmentador.tokenize(parrafo2)
print(segmentos2)

y ahora la salida ya está mejor segmentada:

```
['Los modelos geométricos, construidos en el espacio de instancias y que pueden tener una, dos o múltiples dimensiones.', 'Si hay un borde de decisión lineal entre las clases, se dice que los datos son linealmente separables.']
['El señor J.P. revisó el módulo de la A.P.I. en el servidor a las 09:30h antes de la reunión con el equipo de R.I.', 'Luego la Dra. M.L. ejecutó las pruebas del nuevo O.S. sobre la G.P.U. principal a las 17:45h, justo antes del cierre de las instalaciones.']
```

En el fichero adjunto puedes encontrar una lista de abreviaciones. Ahora lo que haremos será cargar el `spanish_propio.pickle` que hemos entrenado y modificarlo añadiendo la lista de abreviaturas y acrónimos del fichero. Guardaremos este nuevo segmentador como `spanish_propio_abrev.pickle`.

In [ ]:
import nltk.data
import codecs
import pickle

# Cargar tokenizador desde el archivo .pkl
with open("/content/spanish_propio.pickle", "rb") as f:
    segmentador = pickle.load(f)

archivo_abreviaturas = codecs.open("/content/abrev-es.txt", "r", encoding="utf-8")
abreviaturas_extra = []
for abreviatura in archivo_abreviaturas.readlines():
    abreviatura = abreviatura.rstrip()
    abreviaturas_extra.append(abreviatura)
segmentador._params.abbrev_types.update(abreviaturas_extra)

# Guarda el tokenizador entrenado en un archivo
with open("spanish_propio_abrev.pickle", "wb") as f:
    pickle.dump(segmentador, f)

---

#**Ejercicio:**

Hacer los propio con un segmentador para inglés, añadiendo un listado de abreviaturas


---

# 6️⃣ Análisis de frecuencia y ley de Zipf

En este apartado vamos a hacer algún cálculo sencillo sobre corpus: frecuencias absolutas y frecuencias relativas, distribuciones de frecuencia y a encontrar las colocaciones más frecuentes de un corpus.


## Frecuencia absoluta

Entendemos por frecuencia absoluta el número total de veces que aparece una determinada unidad léxica en nuestro corpus.

El cálculo de la frecuencia absoluta de una palabra es sencillo: podemos utilizar un diccionario para poner como clave las palabras e ir incrementando el valor del diccionario cada vez que aparece la palabra. En el siguiente programa podemos ver una implementación sencilla de esta idea:

In [ ]:
import nltk
from nltk.tokenize import RegexpTokenizer

# Cargar tokenizador desde el archivo .pkl
with open("/content/spanish_propio_abrev.pickle", "rb") as f:
    segmentador = pickle.load(f)
    frases_es = segmentador.tokenize(texto_total_es)
    tokenizador = RegexpTokenizer(r"\w+-\d+|\w+|[^\w\s]+")
    tokens_por_frase = [tokenizador.tokenize(s) for s in frases_es]

    # Aplanar lista
    tokens = [tok for frase in tokens_por_frase for tok in frase]

    # Calcular frecuencias
    frecuencia = {}
    for token in tokens:
        frecuencia[token] = frecuencia.get(token, 0) + 1

    # Mostrar resultados ordenados
    for palabra, freq in sorted(frecuencia.items(), key=lambda x: x[1], reverse=True):
        print(f"{freq}\t{palabra}")

    print("Número de palabras:", len(tokens))
    print("Primeras 20 palabras:", tokens[:20])

y por pantalla nos mostrará las palabras y las frecuencias. NLTK proporciona una función FreqDist que facilita mucho el cálculo de frecuencias.

In [ ]:
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk import FreqDist

# Cargar segmentador desde el archivo .pkl para luego tokenizar cada oración
with open("/content/spanish_propio_abrev.pickle", "rb") as f:
    segmentador = pickle.load(f)
    frases_es = segmentador.tokenize(texto_total_es)
    tokenizador = RegexpTokenizer(r"\w+-\d+|\w+|[^\w\s]+")
    tokens_por_frase_es = [tokenizador.tokenize(s) for s in frases_es]

    # Aplanar lista
    tokens_es = [tok for frase in tokens_por_frase_es for tok in frase]

    frequencia = FreqDist(tokens_es)

    for mc in frequencia.most_common():
        print(mc)

## Frecuencia relativa

La frecuencia absoluta de una palabra en un determinado corpus no nos da información real sobre si la palabra es muy frecuente o no, porque esto dependerá del tamaño del corpus. Que una palabra aparezca una cierta cantidad de veces en nuestro corpus no nos dice nada, puesto que si el corpus es muy grande quizás este valor de frecuencia sea pequeño.

La frecuencia relativa de una palabra en un corpus es el número a veces que aparece dividida por el número total de palabras en el corpus.

FreqDist nos facilita mucho el cálculo de la frecuencia relativa puesto que la podemos consultar con el método freq(). Ahora guardamos en el fichero frequencias.txt las palabras ordenadas por frecuencia y mostramos la frecuencia absoluta y la relativa de cada palabra:

In [ ]:
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk import FreqDist
import codecs

# Cargar tokenizador desde el archivo .pkl
with open("/content/spanish_propio_abrev.pickle", "rb") as f:
    segmentador = pickle.load(f)
    segmentos = segmentador.tokenize(texto_total_es)
    tokenizador = RegexpTokenizer(r"\w+-\d+|\w+|[^\w\s]+")
    tokens_por_frase_es = [tokenizador.tokenize(s) for s in segmentos]

    # --- Aplanar lista ---
    tokens_es = [
        t.lower()
        for frase in tokens_por_frase_es
        for t in frase
        if t.isalpha()  # solo palabras, descarta puntuación, números, etc.
    ]

    # --- Calcular frecuencias ---
    frecuencia = FreqDist(tokens_es)

    # --- Guardar en archivo ---
    with codecs.open("/content/frecuencias.txt", "w", encoding="utf-8") as salida:
        for palabra, frecuencia_absoluta in frecuencia.most_common():
            frecuencia_relativa = frecuencia.freq(palabra)
            cadena = (
                str(frecuencia_absoluta)
                + "\t"
                + str(frecuencia_relativa)
                + "\t"
                + palabra
            )
            print(cadena)

Y la salida (mostramos solo las 10 primeras):
```
1323	0.07862364057764307	de
694	0.04124324003090272	la
461	0.02739644618767457	en
442	0.026267308492304034	el
437	0.025970166993522316	y
397	0.023593035003268557	que
302	0.01794734652641588	a
298	0.017709633327390503	los
223	0.013252510845664705	un
206	0.012242229749806858	se
```
Hay que darse cuenta que queremos aplanar y convertir todo a minúsculas, dejando solo tokens alfabéticos.

## La ley de Zipf

La Ley de Zipf afirma que dado un corpus en un idioma, la frecuencia de una palabra tiende a ser inversamente proporcional a su rango en la lista global de palabras  (rank) ordenadas por frecuencia decreciente.

Esta característica está descrita por la [ley de Zipf](https://en.wikipedia.org/wiki/Zipf%27s_law), formulada por el lingüista estadounidense [George Kingsley Zipf] (https://en.wikipedia.org/wiki/George_Kingsley_Zipf).

Sea $n$ el número de palabras en un corpus. Para un exponente fijo $s>0$ (generalmente cercano a $1$), la ley de Zipf estima que la frecuencia de la palabra más frecuente de rango $k$ se aproxima mediante:

$$
f(k) = \frac{(1/k^s)}{ \sum_{i=1}^n (1 / i^s)}
$$

O lo que es lo mismo, con su ley, Zipf afirma que hay una constante $k$ que multiplicando la frecuencia de cualquier palabra por su posición en la tabla (rank):

$$k = f · r$$

En el programa siguiente evaluamos la ley de Zipf con las 50 palabras más frecuentes partiendo de los textos extraídos de la Wikipedia (lo normal sería hacerlo a partir de un corpus):






In [ ]:
def plot_most_common(tokens, top=50, title="Top Words"):
    freq = Counter(tokens)
    most_common = freq.most_common(top)
    words, counts = zip(*most_common)
    plt.figure(figsize=(12, 5))
    plt.bar(words, counts)
    plt.xticks(rotation=45)
    plt.title(title)
    plt.show()
    return freq


freq_es = plot_most_common(tokens_es, title="Top 50 words - Español")

El gráfico generado por ese código muestra las 50 palabras más frecuentes en el conjunto de tokens analizado del texto de la wikipedia escrapeado en español. En el eje horizontal se ven las palabras, y en el eje vertical su número de ocurrencias. Permite identificar rápidamente cuáles son los términos más repetidos, destacando patrones de vocabulario, posibles stopwords y palabras clave relevantes del texto. La visualización ayuda a tener una visión general de la frecuencia de uso de las palabras dentro del texto.

A continuación vamos a generar el gráfico que permite visualizar la distribución típica de palabras en un idioma, mostrando cómo unas pocas palabras concentran la mayoría de ocurrencias y la frecuencia decae rápidamente a medida que aumenta el rango, evidenciando el patrón que describe la ley de Zipf.

In [ ]:
def zipf_law_plot(freq, title="Zipf Law"):
    sorted_freq = sorted(freq.values(), reverse=True)
    ranks = range(1, len(sorted_freq) + 1)
    plt.figure(figsize=(10, 5))
    plt.loglog(ranks, sorted_freq, marker=".")
    plt.title(title)
    plt.xlabel("Rank")
    plt.ylabel("Frequency")
    plt.show()


zipf_law_plot(freq_es, "Ley de Zipf - Español")

En el eje horizontal se muestra el rango de cada palabra (ordenadas de la más frecuente a la menos frecuente) y en el eje vertical su frecuencia de aparición. Al usar escala logarítmica en ambos ejes (log-log), se puede apreciar que las palabras más frecuentes aparecen muchas veces, mientras que la gran mayoría de palabras aparecen pocas veces.

# 7️⃣ Stemming

En Recuperación de Información, uno de los objetivos es agrupar variantes de la misma palabra para mejorar la búsqueda y el análisis de documentos.

Por ejemplo, `“correr”`, `“corriendo”` y `“corrí”` representan la misma idea. Si no aplicamos stemming, cada variante se tratará como una palabra diferente, lo que reduce la eficacia de la recuperación.

El stemming es el proceso de reducir las palabras a su raíz o “stem”, de manera que todas las variantes compartan la misma representación.

Vamos a probar a aplicar un stemmer sobre la salida del tokenizador que hemos creado previamente.

In [ ]:
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.stem import SnowballStemmer

# Crear stemmers
stemmer_es = SnowballStemmer("spanish")

# Cargar tokenizador desde el archivo .pkl
with open("/content/spanish_propio_abrev.pickle", "rb") as f:
    segmentador = pickle.load(f)
    segmentos = segmentador.tokenize(texto_total_es)
    tokenizador = RegexpTokenizer(r"\w+-\d+|\w+|[^\w\s]+")
    tokens_por_frase_es = [tokenizador.tokenize(s) for s in segmentos]

    # --- Aplanar lista ---
    tokens_es = [
        t.lower() for frase in tokens_por_frase_es for t in frase if t.isalpha()
    ]

    # Aplicar stemming
    stems_es = [stemmer_es.stem(token) for token in tokens_es]
    print("Stems español:", stems_es)

La salida generada es la siguiente:

```
['la', 'inteligent', 'artificial', 'abrevi', 'ia', 'en', 'el', 'context', ...]
```

# 8️⃣ Stopwords

A la vista de la anteriore salida, quizás sería bueno después de tokenizar, aplicaremos un filtrado de stopwords, que son palabras muy frecuentes y poco informativas como `“el”`, `“la”`, `“de”` o `“y”`. Esto nos permite centrarnos en términos relevantes para el análisis y evitar ruido en tareas de recuperación de información o análisis de frecuencia.



In [ ]:
import nltk
from nltk.tokenize import RegexpTokenizer

nltk.download("stopwords")

# Cargar tokenizador desde el archivo .pkl
with open("/content/spanish_propio_abrev.pickle", "rb") as f:
    segmentador = pickle.load(f)
    segmentos = segmentador.tokenize(texto_total_es)
    tokenizador = RegexpTokenizer(r"\w+-\d+|\w+|[^\w\s]+")
    tokens_por_frase_es = [tokenizador.tokenize(s) for s in segmentos]
    stops = set(stopwords.words("spanish"))
    # --- Aplanar lista ---
    tokens_es = [
        t.lower()
        for frase in tokens_por_frase_es
        for t in frase
        if t.isalpha() and t.lower() not in stops  # descartamos stopwords
    ]

In [ ]:
freq_es = plot_most_common(tokens_es, title="Top 50 words - Español")

El gráfico generado por ese código muestra las 50 palabras más frecuentes en el conjunto de tokens analizado del texto de la wikipedia escrapeado en español, pero eliminando las stopwords.

Finalmente aplicamos el preprocesado completo: segmentación, tokenización, eliminación de stopwords y stemming.

In [ ]:
import nltk
from nltk.tokenize import RegexpTokenizer

nltk.download("stopwords")
from nltk.stem import SnowballStemmer

# Crear stemmers
stemmer_es = SnowballStemmer("spanish")

# Cargar tokenizador desde el archivo .pkl
with open("/content/spanish_propio_abrev.pickle", "rb") as f:
    segmentador = pickle.load(f)
    segmentos = segmentador.tokenize(texto_total_es)
    tokenizador = RegexpTokenizer(r"\w+-\d+|\w+|[^\w\s]+")
    tokens_por_frase_es = [tokenizador.tokenize(s) for s in segmentos]

    stops = set(stopwords.words("spanish"))
    # --- Aplanar lista ---
    tokens_es = [
        t.lower()
        for frase in tokens_por_frase_es
        for t in frase
        if t.isalpha() and t.lower() not in stops  # descartamos stopwords
    ]

    # Aplicar stemming
    stems_es = [stemmer_es.stem(token) for token in tokens_es]
    print("Stems español:", stems_es)

---

#**Ejercicio:**

Repetir el proceso para el inglés.


---


#**Ejercicio:**

Hacer un motor de búsqueda con las entradas de la wikipedia descargada usando para indexar los textos preprocesados en español.

---